<a href="https://colab.research.google.com/github/TomographicImaging/gVXR-Tutorials/blob/main/notebooks/magnification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
#
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:    Franck Vidal (UKRI-STFC)

![gVXR](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/Logo-transparent-small.png?raw=1)

# Magnification, SDD, SOD, ODD and inverse square law.

In this notebook, we explore
1. how to exploit the magnification to compute the pixel size in the object plane rather
than flat panel plane. This is useful to show the actual size of features, e.g. how big an object.
2. how the source-to-detector distance, at a constant magnification, affects the image quality in terms of noise.
This is due to the ["inverse square law"](https://en.wikipedia.org/wiki/Inverse-square_law).
3. the impact of sample material, noise and polychromatism on CT reconstructions.

![illustration](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/illustration.jpg?raw=1)

<div class="alert alert-block alert-warning">
    <b>Note:</b> Make sure the Python packages are already installed. See <a href="../README.md">README.md</a> in the root directory of the repository. If you are running this notebook from Google Colab, please run the cell below to install the required packages.
</div>

Install condacolab if needed

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

Check if Conda is working well on Google Colab if needed

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import condacolab
    condacolab.check()

Install CIL using Conda on Google Colab if needed

In [ ]:
if IN_COLAB:
    !conda install -y -c conda-forge -c https://software.repos.intel.com/python/conda -c ccpi cil=24.3.0 ipp=2021.12 tigre

Install other system packages and gVXR


In [ ]:
if IN_COLAB:
    !apt-get install libnvidia-gl-580
    !apt-get install \
        libxcb-res0 \
        libxcb-ewmh2 \
        libxcb-composite0 \
        libxcb-cursor0 \
        libxcb-xinerama0 \
        libxcb-keysyms1 \
        libxcb-icccm4 \
        libxcb-xkb1

    !pip install gvxr k3d

## Aims of this session

1. Understand the notion of pixel size in the context of a cone-beam geometry and how the magnification affects it.
2. Become familiar with the inverse square law and how it affects the noise in X-ray images.
3. Identify sources of imaging artefacts (cupping artefacts, streaks, and noise) in reconstructed CT reconstructions.

![Screenshot of the 3D environment using K3D](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/notebooks/output_data/magnification/k3d_screenshot-cropped.png?raw=1)

## Import packages

- `os` to create the output directory if needed
- `matplotlib` to show 2D images
- `SimpleITK` to load the DICOM file
- `gvxr` to simulate X-ray images
- `gvxr2json` to store details of a simulation in a JSON file
- Various `cil` packages for CT reconstruction

In [ ]:
import os # Create the output directory if necessary
import numpy as np # Who does not use Numpy?

import matplotlib.pyplot as plt # Plotting
plt.style.use('tableau-colorblind10')

#  CT simulation
from gvxrPython3 import gvxr, gvxr2json
from gvxrPython3.JSON2gVXRDataReader import *

# CT reconstruction
from cil.plugins.astra import FBP
from cil.io import TIFFWriter

from cil.processors import TransmissionAbsorptionConverter
from cil.utilities.display import show_geometry

## Getting the data ready

Where to save the data.

In [ ]:
output_path = "./output_data/magnification"
if not os.path.exists(output_path):
    os.makedirs(output_path)


## 1. Set the simulation environment


In [ ]:
# Create an OpenGL context
print("Create an OpenGL context")
gvxr.createOpenGLContext(-1, 4, 6, 41)

Set the device geometrical properties.

In [ ]:
# Create a source
print("Set up the beam")
gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
#  For a parallel source, use gvxr.useParallelBeam()

# Set its spectrum, and flux
gvxr.setVoltage(200, "kV")
gvxr.setmAs(140 / 1000 * 0.1) # Current 140 uA, exposure time 0.1 second
gvxr.setBinTotalEnergyCutoff(1, "keV")
gvxr.setEnergyBinSize(2, "keV")

# Enable Poisson noise
gvxr.enablePoissonNoise()

# Set up the detector
print("Set up the detector")
gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)

# We use a downsampling factor to speed up calculation
downsampling_factor = 4
gvxr.setDetectorNumberOfPixels(2048 // downsampling_factor, 2048 // downsampling_factor)
imager_pixel_spacing_in_um = np.array([150.0 * downsampling_factor, 150.0 * downsampling_factor])
gvxr.setDetectorPixelSize(*imager_pixel_spacing_in_um, "um")

In [ ]:
# Locate the sample STL file from the package directory
path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"

# Load the sample data
if not os.path.exists(fname):
    raise IOError(fname)

print("Load the mesh data from", fname)
gvxr.loadMeshFile("sample", fname, "mm")

print("Move ", "sample", " to the centre")
gvxr.moveToCentre("sample")

# Material properties
print("Set ", "sample", "'s material")

# We will re-run this notebook with aluminium as the sample material to explore how the sample material, noise and
# polychromatism impact the quality of the CT reconstructions
# Aluminium (Z number: 13, symbol: Al)
# gvxr.setElement("sample", 13)
# gvxr.setElement("sample", "Al")

# Ice
gvxr.setCompound("sample", "H2O")
gvxr.setDensity("sample", 0.9167, "g/cm3")
gvxr.setDensity("sample", 0.9167, "g.cm-3")

In [ ]:
material_label = gvxr.getMaterialLabel("sample")
material_label = material_label.replace("Element: ", "")
material_label = material_label.replace("Compound: ", "")
material_label = material_label.replace("Mixture: ", "")
print(material_label)

## 2. Using fairly small source-detector distance

We will generate an X-ray image, and compute the system magnification to find the corresponding pixel size. We will also
calculate its PSNR and
simulate a CT scan
acquisition.

Compute the X-ray image.

In [ ]:
xray_image_smaller_SDD = np.array(gvxr.computeXRayImage(), dtype=np.single) / gvxr.getTotalEnergyWithDetectorResponse()

Display the X-ray image with the axes using centimetres as a unit.

In [ ]:
# Conversion um to cm
imager_pixel_spacing_in_cm = imager_pixel_spacing_in_um / 10000

In [ ]:
vmin=0.0
vmax=1.0

fig = plt.figure()
plt.imshow(xray_image_smaller_SDD, cmap="gray", vmin=vmin, vmax=vmax,
                             extent=(
                                 0,(xray_image_smaller_SDD.shape[1]-1)*imager_pixel_spacing_in_cm[0],
                                 0,(xray_image_smaller_SDD.shape[0]-1)*imager_pixel_spacing_in_cm[1]))
plt.title("Digital radiograph")
plt.xlabel("Pixel position\n(in cm)")
plt.ylabel("Pixel position\n(in cm)")
plt.colorbar()
plt.show()

---
## Task:

Write down the sample size and compare it with the output of the cell that contains `loadMeshFile` or retrieve the
bounding box size using the cell below. Is it what you expected? Why not?

In [ ]:
# Retrieve the bounding box
bbox = gvxr.getNodeAndChildrenBoundingBox("root", "cm")

# Compute its size along the three axis
bbox_size = [bbox[3] - bbox[0], bbox[4] - bbox[1], bbox[5] - bbox[2]]

# Sort the array from large to small
bbox_size.sort(reverse=True)

print("Bounding box [in cm]", bbox_size)

## The notion of magnification

Retrieve the source-to-object distance (SOD) and source-to-detector distance (SDD).

In [ ]:
initial_sod = gvxr.getSOD("mm")
initial_sdd = gvxr.getSDD("mm")
initial_odd = gvxr.getODD("mm")

print("Source-Object Distance:", initial_sod, "mm")
print("Source-Detector Distance:", initial_sdd, "mm")
print("Object-Detector Distance:", initial_odd, "mm")

Compute the magnification

$$magnification = \frac{SDD}{SOD}$$

with $SDD$ the source to detector distance and $SOD$ the source to object distance.

---
## Task:

Using the calculator, compute the magnification and run the cell below to check your value.

<div class="alert alert-block alert-warning">
    <b>Note:</b> To myself, add a diagram.
</div>

In [ ]:
magnification = gvxr.getMagnification()
print("Magnification:", magnification)

Compute the pixel size in the object plane.

In [ ]:
spacing_in_cm = (imager_pixel_spacing_in_um / magnification) / 10000.0
print("Pixel size:", spacing_in_cm, "mm")

Display the experimental image using the "pixel size" in the object plane rather than the detector plane.

In [ ]:
fig = plt.figure()
plt.imshow(xray_image_smaller_SDD, cmap="gray", vmin=vmin, vmax=vmax,
           extent=(0,(xray_image_smaller_SDD.shape[1]-1)*spacing_in_cm[0],
                   0,(xray_image_smaller_SDD.shape[0]-1)*spacing_in_cm[1]))
plt.title("Digital radiograph of section of electric cable from my garage")
plt.xlabel("Pixel position\n(in cm)")
plt.ylabel("Pixel position\n(in cm)")
plt.colorbar()
plt.show()

---
## Task:

1. Deduct the size of the sample in mm.
2. Is it more inline we the reality?

## Perform a CT scan acquisition

Select the number of projections, make it at most 1000.

In [ ]:
number_of_projections = min(1000, gvxr.getOptimalNumberOfProjectionsCT())
print(f"Number of projections: {number_of_projections}")

Perform the actual simulations.

In [ ]:
rotation_centre_in_mm = gvxr.getNodeAndChildrenBoundingBoxCentre("sample", "mm")
rotation_axis = gvxr.getDetectorUpVector()

simulation_string_ID = "material_" + material_label + "-smaller_SDD"
gvxr.computeCTAcquisition(
    os.path.join(output_path, "projections-" + simulation_string_ID), # The path where the X-ray projections will be
    # saved
    "", # The path where the screenshots will be saved
    number_of_projections, # The total number of projections to simulate
    0, # The rotation angle corresponding to the first projection
    False, # A boolean flag to include or exclude the last angle
    360, # The rotation angle corresponding to the last projection
    50, # The number of white images used to perform the flat-field correction
    *rotation_centre_in_mm, # The location of the rotation centre
    "mm", # The corresponding unit of length
    *rotation_axis # The rotation axis
)

JSON_fname = os.path.join(output_path, simulation_string_ID + ".json")
gvxr2json.saveJSON(JSON_fname)

## 3. The inverse square law and its relationship with noise

The inverse square law explains that a given physical quantity or intensity decreases in proportion to the square of
the distance from its source. As you move farther from a point source, such as X‑rays, the radiation spreads over a
wider area, resulting in a rapid reduction in intensity and an increase in noise.

![Illustration of the inverse square law](https://upload.wikimedia.org/wikipedia/commons/2/28/Inverse_square_law.svg)
By <a href="//commons.wikimedia.org/wiki/User:Borb" title="User:Borb">Borb</a>, <a href="https://creativecommons.org/licenses/by-sa/3.0" title="Creative Commons Attribution-Share Alike 3.0">CC BY-SA 3.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=3816716">Link</a>

As previously, we will generate an X-ray image, and find the corresponding pixel
size. The major change is significant increase in source-detector distance to see the effect of noise.
This is why we will also calculate its PSNR and
simulate a CT scan acquisition.

Compute the PSNR of the current image.

In [ ]:
PSNR_smaller_SDD = gvxr.getPSNR()
print("PSNR, smallest SDD:", PSNR_smaller_SDD, "dB")

We are going to increase the SDD by a factor 2 without changing the magnification. Remember, $magnification = \frac{SDD}{SOD}$
Therefore we have:

$$SOD = \frac{SDD}{magnification}$$

In [ ]:
new_sdd = 2500 # i.e. 2.5 m
new_sod = new_sdd / magnification
new_odd = new_sdd - new_sod

print("Old SDD:", initial_sdd, "mm")
print("New SDD:", new_sdd, "mm")
print()
print("Old SOD:", initial_sod, "mm")
print("New SOD:", new_sod, "mm")
print()
print("Old ODD:", initial_odd, "mm")
print("New ODD:", new_odd, "mm")

In [ ]:
gvxr.setSourcePosition(-new_sod,  0.0, 0.0, "mm")
gvxr.setDetectorPosition(new_odd, 0.0, 0.0, "mm")

Compute a new image and compare it with the previous one.

In [ ]:
xray_image_larger_SDD = (np.array(gvxr.computeXRayImage(), dtype=np.single) / gvxr
                                     .getTotalEnergyWithDetectorResponse())

Plot them and their diagonal profiles.

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(15,5))

axs[0].set_title("SDD = " + str(initial_sdd) + " mm")
axs[0].imshow(xray_image_smaller_SDD, cmap="gray", vmin=vmin, vmax=vmax,
              extent=(0,(xray_image_smaller_SDD.shape[1]-1)*spacing_in_cm[0],
                      0,(xray_image_smaller_SDD.shape[0]-1)*spacing_in_cm[1]))
axs[0].set_xlabel("Pixel position\n(in cm)")
axs[0].set_ylabel("Pixel position\n(in cm)")

axs[1].set_title("SDD = " + str(new_sdd) + " mm")
axs[1].imshow(xray_image_larger_SDD, cmap="gray", vmin=vmin, vmax=vmax,
              extent=(0,(xray_image_larger_SDD.shape[1]-1)*spacing_in_cm[0],
                      0,(xray_image_larger_SDD.shape[0]-1)*spacing_in_cm[1]))
axs[1].set_xlabel("Pixel position\n(in cm)")
axs[1].set_ylabel("Pixel position\n(in cm)")

axs[2].set_title("Diagonal profiles")
axs[2].plot(np.diag(xray_image_larger_SDD), label="SDD = " + str(new_sdd / 10) + " cm")
axs[2].plot(np.diag(xray_image_smaller_SDD), label="SDD = " + str(initial_sdd / 10 ) + " cm")
axs[2].legend()

plt.show()

Compare their PSNR values (the higher, the better). According to the inverse square law, the larger the SDD, the
lower the number of photons per pixels, i.e. the more noise will be observed.

In [ ]:
PSNR_larger_SDD = gvxr.getPSNR()

print("PSNR, smallest SDD:", PSNR_smaller_SDD, "dB")
print("PSNR, largest SDD:", PSNR_larger_SDD, "dB")

## Evolution of the noise level

We are going to vary the SDD while keeping the magnification constant and plot the PSNR as a function of SDD.

In [ ]:
sdd_set = []
psnr_set = []
for new_sdd in np.linspace(initial_sdd, 2500, 10, True):
    new_sod = new_sdd / magnification
    new_odd = new_sdd - new_sod

    gvxr.setSourcePosition(-new_sod,  0.0, 0.0, "mm")
    gvxr.setDetectorPosition(new_odd, 0.0, 0.0, "mm")

    sdd_set.append(new_sdd / 10) # Conversion mm to cm
    psnr_set.append(gvxr.getPSNR())

In [ ]:
fig = plt.figure()
plt.plot(sdd_set, psnr_set)
plt.xlabel("Source-detector distance\n(in cm)")
plt.ylabel("PSNR\n(in dB)")
plt.show()

Select the number of projections, make it at most 1000.


In [ ]:
number_of_projections = min(1000, gvxr.getOptimalNumberOfProjectionsCT())
print(f"Number of projections: {number_of_projections}")

Perform the actual simulations


In [ ]:
rotation_centre_in_mm = gvxr.getNodeAndChildrenBoundingBoxCentre("sample", "mm")
rotation_axis = gvxr.getDetectorUpVector()

simulation_string_ID = "material_" + material_label + "-larger_SDD"
gvxr.computeCTAcquisition(
    os.path.join(output_path, "projections-" + simulation_string_ID), # The path where the X-ray projections will be
    # saved
    "", # The path where the screenshots will be saved
    number_of_projections, # The total number of projections to simulate
    0, # The rotation angle corresponding to the first projection
    False, # A boolean flag to include or exclude the last angle
    360, # The rotation angle corresponding to the last projection
    50, # The number of white images used to perform the flat-field correction
    *rotation_centre_in_mm, # The location of the rotation centre
    "mm", # The corresponding unit of length
    *rotation_axis # The rotation axis
)

JSON_fname = os.path.join(output_path, simulation_string_ID + ".json")
gvxr2json.saveJSON(JSON_fname)

## Cleaning up

Once we have finished, it is good practice to clean up the OpenGL contexts and windows with the following command. Note that due to the object-oriented programming nature of the core API of gVXR, this step is automatic anyway.

In [ ]:
gvxr.destroy()

---
## Task:

1. Restart the kernel.
2. Use aluminium instead of ice for the sample material.
3. Re-run the notebook.
4. Run the reconstruction notebook called `magnification-reconstruction.ipynb`.
5. Compare the four reconstructed images.
  - Can you see the effect of the noise (i.e. the SDD) on the reconstruction?
  - Can you see the cupping artefacts with the denser material?

---
## Optional tasks:

1. Add an aluminium filter to reduce beam hardening artefacts.
2. Reduce the number of photons much more.
3. Generate cone-beam artefacts.